# M30 · Travel times in a layered crust

**Week 3 · about 9 h (+ 1 h optional)** · before this: M10, M20

> **한국어 요약** 두 가지 성층 지각 모델(kim1983, kim2011)에서 직접파와 선두파의 주시를 계산하는 함수를 직접 만듭니다. 2019년 상주 지진에 대한 여러분의 피킹으로 와다티 도표를 그려 발생 시각과 Vp/Vs 비를 구하고, 잔차로 두 모델을 비교합니다. 마지막으로 TauP를 실행해 여러분의 주시와 비교합니다.

**You will**

- Compute travel times of direct and head waves
- Find the origin time and Vp/Vs from your picks
- Compare two velocity models with your picks
- Run TauP and compare it with your times

**New words:** direct wave (직접파), ray parameter (파선 변수), shooting method (슈팅 방법(사격법)), head wave (선두파), crossover distance (교차 거리), first arrival (최초 도달파 (초동)), Wadati diagram (와다티 도표), residual (잔차) · [all terms](../../../docs/glossary.md)

**Do first, by hand:** [NMSOP-2 EX 11.1](https://doi.org/10.2312/GFZ.NMSOP-2_EX_11.1), before Part 1

**Optional reading:** Shearer (2019), travel times; Havskov, Bormann & Schweitzer (2012), the Wadati diagram

Log any AI help in `work/ai_log/M30.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M30.yaml](../../../.sgtl/criteria/M30.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # your STP download (M03)
KMA_EVENT_ID = "2019009762"               # KMA id of the ML 3.9 Sangju earthquake, 21 July 2019
EVENT_FOLDER = "20190721020418"           # its folder in the download (UTC origin time)
PICKS_FILE = "work/picks/M20_manual_picks.csv"   # your manual picks of M20
MAX_DISTANCE_KM = 100.0                   # Parts 4-6: picks out to this epicentral distance (km)
X_MAX_KM = 200.0                          # travel-time curves out to this distance (km)
CHANGED_DEPTH_KM = 10.0                   # task 5: the one changed parameter of the TauP run (depth, km)
FIGURE_DIR = "figures"                    # your figures go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here

In [ ]:
import sgtl_tutorial as st
S = st.init("M30")

import contextlib
import io
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import UTCDateTime, read
from obspy.geodetics import gps2dist_azimuth

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)

## Part 1 · Two layered models of the crust (about 20 min)

The **travel time** of a wave from the source to a station depends on the **velocity model**, the wave speeds
at each depth. The course has two published models for South Korea, `kim1983` and `kim2011`. Each row is a flat
layer with constant `vp` and `vs` (km/s), from `top_km` down to the next row. The last row has no bottom (a
**half-space**); its top is the **Moho**, where the mantle begins. The cell also reads the Sangju earthquake
from the KMA catalog.

In [ ]:
KIM1983 = st.data.velocity_model("kim1983")
KIM2011 = st.data.velocity_model("kim2011")
for model in (KIM1983, KIM2011):
    print(f"{model.attrs['name']}: {model.attrs['citation']}")
    print(model.to_string(index=False), "\n")

catalog = st.data.kma_catalog()
event = catalog.loc[catalog["event_id"] == KMA_EVENT_ID].iloc[0]
ORIGIN_KMA = UTCDateTime(event["origin_utc"].isoformat())
EV_LAT, EV_LON = float(event["latitude"]), float(event["longitude"])
DEPTH_KM = float(event["depth_km"])
print(f"KMA {KMA_EVENT_ID}: origin {ORIGIN_KMA} (UTC), epicenter {EV_LAT:.2f} N {EV_LON:.2f} E, depth "
      f"{DEPTH_KM:g} km, ML {event['ml']:.1f}")
print(f"source of the catalog: {catalog.attrs.get('source', 'KMA')}")

## Part 2 · Rays through flat layers (about 30 min)

A **ray** is the path a wave follows. The **direct wave** leaves the source upward and reaches the station
without a reflection. Inside a layer the ray is straight. At a boundary it bends by **Snell's law**: $\sin i / v$
is the same on both sides ($i$ is the angle from the vertical, $v$ the speed). So one number stays the same
along the whole ray, the **ray parameter** (s/km):

$$p = \frac{\sin i_j}{v_j}\quad\text{in every layer } j$$

Over the layers above the source (thickness $h_j$), the ray moves sideways $X(p) = \sum_j h_j \tan i_j$ and
takes $T(p) = \sum_j h_j/(v_j \cos i_j)$ seconds. $X$ grows from 0 (straight up) without limit as $p$ nears
1 / (the fastest speed), so each distance has exactly one $p$.

The **shooting method** finds it by trial: fire a ray, see where it lands, adjust $p$. **Bisection** does the
adjusting: halve the interval of $p$ that holds the answer, about 50 times.

### M30-T1 · Direct-wave travel time by shooting · core · about 120 min

Write `direct_time(x_km, depth_km, model, phase="P")`: the travel time (s) of the direct wave to a surface
station `x_km` away, found by shooting on $p$. The docstring has the details.

Example: one layer of 6 km/s, a source 10 km deep and a station 24 km away give 26 / 6 = 4.3333 s.

The check uses the one-layer formula $T = \sqrt{x^2 + z^2}/v$, **Fermat's principle** (no path is quicker than
the ray) and TauP (Part 6).

Not allowed: `obspy.taup`, `pyrocko` and the course package.

In [ ]:
# Before you code: a source 14 km deep and a station 100 km away. Will the direct P arrive earlier in kim1983 or
# in kim2011?
st.predict("M30-T1", """

""")

In [ ]:
def direct_time(x_km, depth_km, model, phase="P"):
    """Travel time (s) of the direct wave from a source at depth_km to a surface station x_km away.

    x_km     : epicentral distance(s) in km (0 or more), one number or a 1-D numpy array
    depth_km : source depth in km (> 0); the direct ray crosses only what lies above the source, and a source
               exactly on a boundary belongs to the layer below
    model    : table like KIM1983 with top_km, vp, vs (km/s); the first top is 0, the last row is a half-space;
               do not change the table
    phase    : "P" (use vp) or "S" (use vs)
    returns  : a float for one distance, a 1-D numpy array of the same length for an array of distances;
               within 5 ms of the true time out to 150 km
    """
    # Step 1: the layers above the source: thickness h_j (the source layer only down to depth_km) and speed v_j.
    # Step 2: for one ray parameter p: sin(i_j) = p v_j; X(p) = sum h_j tan(i_j); T(p) = sum h_j / (v_j cos(i_j)).
    # Step 3: p runs from 0 to 1 / max(v_j). Bisection on p until X(p) = x (for every distance), then T(p).
    # YOUR CODE HERE
    raise NotImplementedError("M30-T1: write your code here")

In [ ]:
st.check("M30-T1", func=direct_time)

Your function on the example of the task, and on both models for a source at the KMA depth. The last column
follows a straight line instead of the ray.

In [ ]:
one_layer = pd.DataFrame({"top_km": [0.0], "vp": [6.0], "vs": [3.5]})
print(f"example of the task: {direct_time(24.0, 10.0, one_layer):.4f} s")
x_show = np.array([0.0, 10.0, 30.0, 50.0, 100.0, 150.0])
table = pd.DataFrame({"x_km": x_show})
for name, model in (("kim1983", KIM1983), ("kim2011", KIM2011)):
    for phase in ("P", "S"):
        table[f"{name} {phase} (s)"] = direct_time(x_show, DEPTH_KM, model, phase)
upper = KIM2011[KIM2011["top_km"] < DEPTH_KM]                    # kim2011's layers above the source (Part 1)
km_in_layer = np.diff(np.append(upper["top_km"].to_numpy(), DEPTH_KM))
mean_slowness = float(np.sum(km_in_layer / upper["vp"].to_numpy())) / DEPTH_KM    # P along a straight line (s/km)
table["straight line in kim2011, P (s)"] = np.hypot(x_show, DEPTH_KM) * mean_slowness
print(f"direct waves from a source at {DEPTH_KM:g} km:")
print(table.round(3).to_string(index=False))

**Explain:** At 100 km, which model gives the earlier direct P, and why? Look at the layer each source sits in.

## Part 3 · Head waves (about 30 min)

A ray going down can meet a faster layer. At the **critical angle**, $\sin i = v_{above}/v_{below}$, it runs
along the boundary in the faster layer and sends energy back up all the way: the **head wave**. Its ray
parameter is $1/v_{below}$. Along the Moho it is called **Pn** (P) or **Sn** (S).

It goes down from the source, runs along the boundary and comes back up to the station. Each **leg** through
a layer is slanted at that layer's angle, $\sin i = v_{layer}/v_{below}$.

The head wave starts at the **critical distance**. Beyond it, its time is a straight line,
$t = x/v_{below} + t_i$ ($t_i$: the **intercept time**). It overtakes the direct wave at the **crossover
distance**. The earliest wave at a station is the **first arrival**, the onset a picker marks.

**Watch out:** in this course head waves are a travel-time idea only. Locate earthquakes with direct P and S
within about 100 km, never with Pn or Sn.

### M30-T2 · Head waves along a boundary · core · about 90 min

Write `head_wave(x_km, depth_km, model, phase="P", interface=-1)`. It returns `(times, critical_km,
crossover_km)` for the head wave along the top of row `interface` (`-1`: below the Moho). Use your
`direct_time` for the crossover. The docstring has the details.

Example: 10 km of 6 km/s over 8 km/s, source 5 km deep, P. Then $\sin i = 0.75$ and the legs add up to
10 + 5 = 15 km, so the critical distance is $15 \tan i$ = 17.01 km. At 100 km the time is
$100/8 + 15 \cos i/6$ = 14.1536 s.

Not allowed: `obspy.taup`, `pyrocko` and the course package.

In [ ]:
# Before you code: for the Sangju source (14 km) in kim1983, which head wave overtakes the direct P first, the one
# along 15 km or along the Moho (32 km), and where?
st.predict("M30-T2", """

""")

In [ ]:
def head_wave(x_km, depth_km, model, phase="P", interface=-1):
    """Head wave along the top of layer `interface`: (times, critical distance, crossover distance).

    x_km      : epicentral distance(s) in km, one number or a 1-D numpy array
    depth_km  : source depth in km
    model     : table with top_km, vp, vs (km/s); do not change it
    phase     : "P" (use vp) or "S" (use vs)
    interface : row number (Python indexing) of the layer along whose top the head wave runs: -1 is the last
                row, the half-space below the Moho in both shipped models (Pn, Sn); 1 is the top of the second row
    returns   : (times, critical_km, crossover_km)
                times        : s, a float for one distance or a 1-D numpy array for an array; NaN before the
                               critical distance
                critical_km  : the critical distance (km)
                crossover_km : beyond this distance (km) the head wave arrives before your direct_time; to better
                               than 0.1 km
                All three are NaN when no head wave can run along that boundary: the source is not above it, or a
                layer above it is at least as fast as the layer below it.
    """
    # Step 1: the boundary: n = the row of the layer below it, z_n its top, v_n its speed; ray parameter 1 / v_n.
    # Step 2: no head wave (all NaN) unless the source is above z_n and every layer above z_n is slower than v_n.
    # Step 3: legs: each layer above z_n is crossed once going up (all of it) and, below the source, once going
    #         down. With sin(i) = v_layer / v_n: critical distance = sum of legs * tan(i);
    #         time = x / v_n + sum of legs * cos(i) / v_layer, from the critical distance on.
    # Step 4: crossover: where your direct_time(x) equals the head-wave time; bisection on the sign of the
    #         difference, between the critical distance and a distance where the head wave is already earlier.
    # YOUR CODE HERE
    raise NotImplementedError("M30-T2: write your code here")

In [ ]:
st.check("M30-T2", func=head_wave)

Your two functions for the Sangju source depth: every boundary below the source, in both models, for P and S.

In [ ]:
rows = []
for name, model in (("kim1983", KIM1983), ("kim2011", KIM2011)):
    for n in range(1, len(model)):
        for phase in ("P", "S"):
            t100, x_crit, x_cross = head_wave(100.0, DEPTH_KM, model, phase, n)
            rows.append({"model": name, "boundary (km)": model["top_km"].iloc[n], "phase": phase,
                         "critical (km)": x_crit, "crossover (km)": x_cross, "head wave at 100 km (s)": t100,
                         "direct at 100 km (s)": direct_time(100.0, DEPTH_KM, model, phase)})
crossovers = pd.DataFrame(rows)
print(f"source depth {DEPTH_KM:g} km (NaN: no head wave along that boundary for this source)")
print(crossovers.round(2).to_string(index=False))

x_grid = np.linspace(0.0, X_MAX_KM, 401)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0), sharey=True)
for ax, (name, model) in zip(axes, (("kim1983", KIM1983), ("kim2011", KIM2011))):
    ax.plot(x_grid, direct_time(x_grid, DEPTH_KM, model, "P") - x_grid / 8.0, color="k", lw=1.6, label="Direct P")
    below = [n for n in range(1, len(model)) if model["top_km"].iloc[n] > DEPTH_KM]
    for k, (n, color) in enumerate(zip(below, ("tab:green", "tab:purple", "tab:orange"))):
        t_head, _, x_cross = head_wave(x_grid, DEPTH_KM, model, "P", n)
        if np.isfinite(t_head).any():
            ax.plot(x_grid, t_head - x_grid / 8.0, color=color, lw=1.3,
                    label=("First", "Second", "Third")[k] + " boundary below the source: head wave")
            ax.axvline(x_cross, color=color, lw=0.8, ls="--")
    ax.axvspan(MAX_DISTANCE_KM, X_MAX_KM, color="0.92", lw=0, zorder=0)
    tops_below = ", ".join(f"{model['top_km'].iloc[n]:g}" for n in below)
    ax.set_title(f"{name}: boundaries below the source at {tops_below} km", fontsize=10)
    ax.set_xlabel("Epicentral distance (km)")
axes[0].set_ylabel("Travel time - distance / 8 km/s (s)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "head_waves.png", dpi=150)

The figure shows the **reduced travel time**: the time minus distance / (8 km/s). A wave at 8 km/s would be
flat, so small differences show. Dashed lines mark your crossover distances; the grey band lies beyond 100 km.

**Explain:** In kim1983, which wave is the first P at 70-100 km, and why does it overtake the direct P so early
for this source?

In [ ]:
# Optional explorer E4 (not checked): change the layers and the source depth; your own two functions draw.
st.widgets.traveltime_explorer("kim1983", direct=direct_time, head=head_wave, depth_km=DEPTH_KM)

## Part 4 · The Wadati diagram from your own picks (about 30 min)

In M20 you picked the P and S **arrival times** of the Sangju earthquake. The cells below read your pick file.
If they find no file, finish the picking task of M20 first.

In [ ]:
PICKS_PATH = st.data.repo_path(PICKS_FILE)
SAC_DIR = st.data.root() / DATASET / "stp_download" / "SAC" / EVENT_FOLDER
print("pick file:", PICKS_FILE, "| found" if PICKS_PATH.is_file() else "| NOT FOUND: run the picking task of M20")

In [ ]:
def read_picks(path, event_ids=(KMA_EVENT_ID, EVENT_FOLDER)):
    """Your pick file as a table: station (NET.STA), phase (P or S) and time (an obspy UTCDateTime, in UTC).

    It accepts the usual spellings of the columns (station or network + station or seed_id; phase; time_utc or
    time) and, when the file has an event column, keeps the rows of this earthquake.
    """
    raw = pd.read_csv(path, dtype=str).rename(columns=lambda c: c.strip().lower())
    when = next((c for c in ("time_utc", "time", "pick_time", "utc", "arrival_time", "arrival_utc") if c in raw),
                None)
    if when is None or "phase" not in raw:
        raise ValueError(f"{path}: need a phase column and a time column (time_utc), found {list(raw.columns)}")
    if "station" in raw:
        names = raw["station"].str.strip()
        if "network" in raw:
            names = names.where(names.str.contains(".", regex=False), raw["network"].str.strip() + "." + names)
    else:
        seed = raw["seed_id"] if "seed_id" in raw else raw["trace_id"]
        names = seed.str.split(".").str[:2].str.join(".")
    keep = pd.Series(True, index=raw.index)
    for col in ("event_id", "event", "utc_id", "event_folder"):
        if col in raw and raw[col].isin(event_ids).any():
            keep = raw[col].isin(event_ids)
    table = pd.DataFrame({"station": names, "phase": raw["phase"].str.strip().str.upper().str[0],
                          "time": [UTCDateTime(t) for t in raw[when].str.strip()]})[keep.to_numpy()]
    return table[table["phase"].isin(["P", "S"])].reset_index(drop=True)


picks = read_picks(PICKS_PATH) if PICKS_PATH.is_file() else None
if picks is not None:
    first = picks.sort_values("time").groupby(["station", "phase"]).first()["time"].unstack("phase")
    pairs = first.dropna()
    tp_s = np.array([t - ORIGIN_KMA for t in pairs["P"]])       # seconds after the KMA origin time
    ts_s = np.array([t - ORIGIN_KMA for t in pairs["S"]])
    print(f"{len(picks)} picks; {len(first)} stations, {len(pairs)} with both P and S")
    print(pd.DataFrame({"station": pairs.index, "P after KMA origin (s)": tp_s, "S after KMA origin (s)": ts_s,
                        "S-P (s)": ts_s - tp_s}).round(3).to_string(index=False))

At one station P arrives at $t_P = t_0 + R/V_P$ and S at $t_S = t_0 + R/V_S$ ($t_0$: the **origin time**,
$R$: the distance to the hypocenter). Remove $R$:

$$t_S - t_P = \left(\frac{V_P}{V_S} - 1\right)(t_P - t_0)$$

So the **S-P time** against the P time is a straight line, the **Wadati diagram**. Its slope is Vp/Vs - 1, and
the S-P time is zero at the origin time. No velocity model is needed. Fit the line by **least squares**: choose
the slope and the intercept that make the sum of the squared misfits smallest.

### M30-T3 · The Wadati diagram · core · about 60 min

Write `wadati_fit(tp_s, ts_s)`. Fit $y = kx + c$ by least squares with $x = t_P$ and $y = t_S - t_P$, and
return `(t0_s, vp_vs)`: where the line reaches $y = 0$, and $1 + k$.

Example: tp = 10, 20, 30 s and ts = 17, 34, 51 s give (0.0, 1.7).

The check also tries your own Sangju picks (P and S at 5 or more stations, times in UTC) against loose physical
limits.

Not allowed: line-fitting libraries such as `numpy.polyfit` and `scipy` (the docstring lists them). Use means
and sums.

In [ ]:
# Before you code: KMA gives the origin time to the whole second. How far from 02:04:18 do you expect your Wadati
# origin time to be?
st.predict("M30-T3", """

""")

In [ ]:
def wadati_fit(tp_s, ts_s):
    """Origin time and Vp/Vs from a least-squares line through the Wadati diagram.

    tp_s, ts_s : 1-D arrays (or lists) of equal length (2 or more): the P and S arrival times at the same
                 stations in the same order, in s after one reference moment of your choice
    returns    : (t0_s, vp_vs), two floats: t0, where the fitted line y = k x + c (x = tp, y = ts - tp, ordinary
                 least squares) reaches y = 0, in s after the same moment; and Vp/Vs = 1 + k
    Not allowed here: numpy.polyfit (also numpy.ma.polyfit), numpy.polynomial, numpy.linalg.lstsq, all of scipy,
    sklearn, statsmodels, torch, xarray, obspy.signal and the course package. Write the line yourself.
    """
    # Step 1: x = tp, y = ts - tp. A straight line y = k x + c.
    # Step 2: least squares: k = sum((x - mean x)(y - mean y)) / sum((x - mean x)^2), c = mean y - k mean x.
    # Step 3: y = 0 at x = -c / k (the origin time); Vp/Vs = 1 + k.
    # YOUR CODE HERE
    raise NotImplementedError("M30-T3: write your code here")

In [ ]:
st.check("M30-T3", func=wadati_fit, picks=picks, event=KMA_EVENT_ID)

In [ ]:
t0_s, vp_vs = wadati_fit(tp_s, ts_s)
ORIGIN_WADATI = ORIGIN_KMA + t0_s
spread = (ts_s - tp_s) - (vp_vs - 1.0) * (tp_s - t0_s)
print(f"your Wadati fit, {tp_s.size} stations: Vp/Vs = {vp_vs:.3f}, origin {ORIGIN_WADATI} "
      f"({t0_s:+.2f} s after the KMA origin time {ORIGIN_KMA})")
print(f"scatter of the points about your line: RMS {np.sqrt(np.mean(spread ** 2)):.3f} s, largest "
      f"{np.abs(spread).max():.3f} s")
for name, model in (("kim1983", KIM1983), ("kim2011", KIM2011)):
    print(f"{name} vp/vs of its layers:", np.round(model["vp"] / model["vs"], 3).tolist())

fig, ax = plt.subplots(figsize=(6.0, 4.2))
line_x = np.array([min(t0_s, 0.0) - 0.5, tp_s.max() + 1.0])
ax.plot(tp_s, ts_s - tp_s, "o", color="tab:blue", ms=5, label="Your picks")
ax.plot(line_x, (vp_vs - 1.0) * (line_x - t0_s), color="k", lw=1.2, label=f"Your line: Vp/Vs = {vp_vs:.3f}")
ax.axvline(0.0, color="tab:red", lw=1.0, ls="--", label="KMA origin time")
ax.plot([t0_s], [0.0], "*", color="gold", mec="k", ms=12, label=f"Your origin time ({t0_s:+.2f} s)")
ax.set_xlabel("P arrival time after the KMA origin time (s)")
ax.set_ylabel("S-P time (s)")
ax.set_title("Wadati diagram, Sangju 2019", fontsize=10)
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.86))
fig.savefig(Path(FIGURE_DIR) / "wadati_diagram.png", dpi=150)

**Explain:** How far is your origin time from KMA's 02:04:18 (printed above), and can the rounding to whole
seconds explain the difference?

## Part 5 · Residuals: how well does a model fit? (about 30 min)

A **residual** is the observed travel time minus the model's time; positive means the wave came late. Your
picks are first arrivals. So compare them with the model's first arrival, the earliest of its direct wave and
its head waves (tasks 1 and 2).

**Watch out:** the crossover distance depends on the source depth. The closer the source is to a boundary below
it, the sooner that boundary's head wave comes first. Your table of Part 3 gives the crossovers for this source:
within 100 km, Pn and Sn never come first.

Four patterns tell different stories:

- **A common shift** (all residuals, P and S alike): the origin time is off.
- **A trend with distance**: the crust is faster (residuals fall with distance) or slower than the model.
- **A split between P and S** after the shift: the model's Vp/Vs is off (compare task 3).
- **Scatter**: picking errors, an epicenter error (KMA gives 0.1 degree, about 10 km), or 3-D structure.

The **RMS residual**, the root of the mean squared residual, sums up the misfit in one number.

The next cell builds the table of observed travel times. It uses your picks within `MAX_DISTANCE_KM` and the
**epicentral distance** of each station, and counts the time from your Wadati origin.

In [ ]:
def station_coordinates(names):
    """(latitude, longitude) of each NET.STA from the SAC headers of your download (any band)."""
    found = {}
    for path in sorted(SAC_DIR.glob("*/*.sac")):
        key = ".".join(path.name.split(".")[1:3])
        if key in names and key not in found:
            head = read(str(path), headonly=True)[0].stats.sac
            found[key] = (float(head.stla), float(head.stlo))
    return found


coords = station_coordinates(set(picks["station"]))
rows = []
for (sta, phase), group in picks.sort_values("time").groupby(["station", "phase"]):
    if sta in coords:
        meters, azimuth, _ = gps2dist_azimuth(EV_LAT, EV_LON, *coords[sta])
        rows.append({"station": sta, "phase": phase, "epi_km": meters / 1000.0, "azimuth": azimuth,
                     "tt_s": group["time"].iloc[0] - ORIGIN_WADATI})
obs = pd.DataFrame(rows)
obs = obs[obs["epi_km"] <= MAX_DISTANCE_KM].sort_values(["epi_km", "phase"]).reset_index(drop=True)
print(f"{len(obs)} travel times ({(obs['phase'] == 'P').sum()} P, {(obs['phase'] == 'S').sum()} S) at "
      f"{obs['station'].nunique()} stations within {MAX_DISTANCE_KM:g} km; counted from your Wadati origin time")
print(obs.round(3).to_string(index=False))

### M30-T4 · Travel-time residuals of two models · core · about 90 min

Write `residual_summary(obs, depth_km, model, direct, head)`. It compares every observed time in `obs` with the
model's first arrival, computed with the two functions it is given (like your `direct_time` and `head_wave`).
It returns a dict with the residuals, the common shift, the RMS after the shift, and the trend with distance
of P and of S. The docstring lists the keys.

Example: two P rows at 10 and 20 km with residuals 0.1 and 0.3 s give `shift_s` 0.2, `rms_s` 0.1, `slope_P`
0.02 s/km and `slope_S` NaN.

Not allowed: the same libraries as in task 3.

In [ ]:
# A colleague compared the two models like this and wrote down the verdict. Run it as it is.
verdict = {}
late_kma = obs["tt_s"].to_numpy() + (ORIGIN_WADATI - ORIGIN_KMA)       # travel times from the KMA origin time
for name, model in (("kim1983", KIM1983), ("kim2011", KIM2011)):
    predicted = np.array([direct_time(x, DEPTH_KM, model, ph) for x, ph in zip(obs["epi_km"], obs["phase"])])
    verdict[name] = float(np.sqrt(np.mean((late_kma - predicted) ** 2)))
    print(f"colleague: {name} RMS residual {verdict[name]:.3f} s (KMA origin time)")
print("colleague's verdict: the model that fits Sangju better is", min(verdict, key=verdict.get))

In [ ]:
# Before you code: do you accept the colleague's verdict, and which model will your residual_summary favor?
st.predict("M30-T4", """

""")

In [ ]:
def residual_summary(obs, depth_km, model, direct, head):
    """Residuals of observed travel times against a model's first arrival, with their common shift, RMS and trends.

    obs      : table with at least phase ("P" or "S"), epi_km and tt_s (observed travel time, s), in any order and
               with any index; do not change it
    depth_km : source depth (km)
    model    : velocity model table (top_km, vp, vs); do not change it
    direct   : function direct(x_km, depth_km, model, phase) -> s, like your direct_time
    head     : function head(x_km, depth_km, model, phase, interface) -> (times, critical_km, crossover_km), like
               your head_wave
    Predict every row with these two functions, the depth and the model given (step 1 below).
    returns  : dict with
        residual_s        1-D numpy array, observed minus predicted, one value per row of obs, in its row order
        shift_s           one shift for all rows (P and S alike) that minimizes the sum of squared residuals
        rms_s             root mean square of the residuals after subtracting shift_s
        slope_P, slope_S  least-squares slope of the residual against epi_km for that phase alone (s/km); NaN when
                          that phase has fewer than 2 rows
    """
    # Step 1: for the rows of each phase, the model's first arrival: the earliest of the direct wave and the head
    #         waves along every boundary (interface 1 to the last row); a NaN head-wave time does not count.
    # Step 2: residual = observed - first arrival, in the row order of obs.
    # Step 3: the shift that minimizes sum((residual - shift)^2) over all rows (set the derivative to zero).
    # Step 4: RMS of (residual - shift); for P rows and for S rows on their own, the least-squares slope of the
    #         residual against epi_km (as in task 3), NaN with fewer than 2 rows.
    # YOUR CODE HERE
    raise NotImplementedError("M30-T4: write your code here")

In [ ]:
st.check("M30-T4", func=residual_summary)

Your function on your Sangju picks, for both models. It runs three times: from your Wadati origin time, from
the KMA origin time, and against the direct wave alone (as in the colleague's cell).

In [ ]:
def no_head_wave(x_km, depth_km, model, phase="P", interface=-1):
    """Called like head_wave, but there is never a head wave: NaN times, NaN distances."""
    return np.full(np.shape(x_km), np.nan), np.nan, np.nan


obs_kma = obs.assign(tt_s=obs["tt_s"] + (ORIGIN_WADATI - ORIGIN_KMA))
is_p = (obs["phase"] == "P").to_numpy()
summary, head_first = {}, {}
for name, model in (("kim1983", KIM1983), ("kim2011", KIM2011)):
    out = residual_summary(obs, DEPTH_KM, model, direct_time, head_wave)
    from_kma = residual_summary(obs_kma, DEPTH_KM, model, direct_time, head_wave)
    alone = residual_summary(obs, DEPTH_KM, model, direct_time, no_head_wave)
    summary[name] = out
    head_first[name] = out["residual_s"] - alone["residual_s"] > 1e-6    # first arrival earlier than the direct
    left = out["residual_s"] - out["shift_s"]
    print(f"{name}: shift {out['shift_s']:+.3f} s, RMS after the shift {out['rms_s']:.3f} s, slope P "
          f"{out['slope_P'] * 100:+.3f} and S {out['slope_S'] * 100:+.3f} s per 100 km; mean after the shift: "
          f"P {left[is_p].mean():+.3f} s, S {left[~is_p].mean():+.3f} s")
    print(f"   from the KMA origin time: shift {from_kma['shift_s']:+.3f} s, RMS after the shift "
          f"{from_kma['rms_s']:.3f} s")
    print(f"   a head wave is the first arrival at {head_first[name][is_p].sum()} of {is_p.sum()} P and "
          f"{head_first[name][~is_p].sum()} of {(~is_p).sum()} S picks; against the direct wave alone: RMS "
          f"{alone['rms_s']:.3f} s, slope P {alone['slope_P'] * 100:+.3f} and S {alone['slope_S'] * 100:+.3f} s "
          "per 100 km")

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), sharey=True)
for ax, name in zip(axes, ("kim1983", "kim2011")):
    out = summary[name]
    for phase, color in (("P", "tab:blue"), ("S", "tab:red")):
        rows_ph = (obs["phase"] == phase).to_numpy()
        x_ph = obs["epi_km"].to_numpy()[rows_ph]
        r_ph = out["residual_s"][rows_ph] - out["shift_s"]
        by_head = head_first[name][rows_ph]
        ax.plot(x_ph[~by_head], r_ph[~by_head], "o", color=color, ms=4, label=f"{phase}: direct wave first")
        ax.plot(x_ph[by_head], r_ph[by_head], "o", mfc="none", color=color, ms=5, label=f"{phase}: head wave first")
        if np.isfinite(out["slope_" + phase]):
            line = np.array([x_ph.min(), x_ph.max()])
            ax.plot(line, r_ph.mean() + out["slope_" + phase] * (line - x_ph.mean()), color=color, lw=1.0)
    ax.axhline(0.0, color="0.5", lw=0.8)
    ax.set_title(f"{name}: RMS {out['rms_s']:.3f} s after the shift", fontsize=10)
    ax.set_xlabel("Epicentral distance (km)")
axes[0].set_ylabel("Residual minus common shift (s)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "residuals_two_models.png", dpi=150)

**Explain:** Which model fits your picks better after the common shift, and why does that differ from the
colleague's verdict?

A locator estimates the origin time together with the hypocenter (M31), so a common shift is absorbed there. What
is left after the shift (a trend, a P-S split, scatter) speaks about the model.

## Part 6 · TauP, a production tool (about 20 min)

**TauP** computes travel times in a layered, spherical Earth; ObsPy contains it as `obspy.taup`. The helper
below writes a TauP model file (`.nd`) from a shipped model, with the global model ak135 below 200 km. Three
calls do the rest:

```python
from obspy.taup.taup_create import build_taup_model    # .nd file -> model file (.npz)
from obspy.taup import TauPyModel                       # TauPyModel(model="<folder>/<name>.npz")
taup.get_travel_times(source_depth_in_km=..., distance_in_degree=..., phase_list=[...])
```

Each arrival has a `.name` and a `.time` (s). In TauP, `P` and `S` leave the source downward; `p` and `s` leave
it upward.

In [ ]:
def write_taup_nd(model, path, join_km=200.0):
    """Write a shipped layered model as a TauP '.nd' file: the layers (constant speeds), 'mantle' at the top of
    the last row, the half-space down to join_km, then the deep Earth of ak135 below join_km."""
    import obspy.taup

    tops = model["top_km"].to_numpy(float)
    bottoms = list(tops[1:]) + [join_km]
    lines = []
    for j, (top, vp, vs) in enumerate(zip(tops, model["vp"], model["vs"])):
        if j == len(tops) - 1:
            lines.append("mantle")
        lines += [f"{top:.3f} {vp:.4f} {vs:.4f} 2.70", f"{bottoms[j]:.3f} {vp:.4f} {vs:.4f} 2.70"]
    ak135 = Path(obspy.taup.__file__).parent / "data" / "ak135f_no_mud.nd"
    for row in ak135.read_text().splitlines():
        parts = row.split()
        if parts in (["outer-core"], ["inner-core"]):
            lines.append(parts[0])
        elif len(parts) >= 4 and float(parts[0]) > join_km:
            lines.append(" ".join(parts[:4]))
    Path(path).write_text("\n".join(lines) + "\n")
    return Path(path)


print(write_taup_nd(KIM1983, Path(RESULT_DIR) / "kim1983.nd").read_text().splitlines()[:9])

### M30-T5 · Travel times from TauP · core · about 60 min

Write `taup_times(model_name, depth_km, x_km)`: the direct P and S travel times that TauP computes for a shipped
model. Build the TauP model from the shipped layers: the helper, then `build_taup_model` into a folder of your
own, then `TauPyModel`. TauP wants the distance in degrees: use an Earth radius of 6371 km.

It returns `(tp, ts)`: the earliest `p` and `s` at each distance, NaN where TauP finds none. The check calls it
twice: both calls must give identical numbers.

In [ ]:
# Before you code: for kim1983 at 14 km, will TauP's direct P be earlier, later or the same as your direct_time?
st.predict("M30-T5", """

""")

In [ ]:
def taup_times(model_name, depth_km, x_km):
    """Direct P and S travel times (s) from TauP for a shipped layered model.

    model_name : "kim1983" or "kim2011"; build its TauP model from the shipped layers (write_taup_nd, then
                 build_taup_model into a folder of your own, then TauPyModel), not a built-in global model
    depth_km   : source depth (km)
    x_km       : 1-D numpy array of epicentral distances (km); TauP wants degrees (Earth radius 6371 km)
    returns    : (tp, ts), two 1-D numpy arrays with one time per distance: the earliest 'p' and the earliest 's'
                 arrival (the direct waves), NaN where TauP finds none; two calls with the same input must give
                 identical numbers
    """
    from obspy.taup import TauPyModel
    from obspy.taup.taup_create import build_taup_model

    # Step 1: write the model file with write_taup_nd into a new folder and build it there (build_taup_model).
    # Step 2: load the built .npz with TauPyModel.
    # Step 3: for every distance: degrees = km / (6371 pi / 180); the earliest 'p' and the earliest 's' arrival.
    # YOUR CODE HERE
    raise NotImplementedError("M30-T5: write your code here")

In [ ]:
st.check("M30-T5", func=taup_times)

The production run: TauP at your stations, an unchanged rerun, then one changed parameter (the source depth),
and the difference from your own function.

In [ ]:
stations = obs.drop_duplicates("station").sort_values("epi_km")
x_st = stations["epi_km"].to_numpy()
base = taup_times("kim1983", DEPTH_KM, x_st)
rerun = taup_times("kim1983", DEPTH_KM, x_st)
changed = taup_times("kim1983", CHANGED_DEPTH_KM, x_st)
same = all(np.array_equal(a, b, equal_nan=True) for a, b in zip(base, rerun))
print(f"unchanged rerun identical to the first run: {same}")
runs = pd.DataFrame({"station": stations["station"].to_numpy(), "epi_km": x_st,
                     "TauP P (s)": base[0],
                     "TauP - yours, P (s)": base[0] - direct_time(x_st, DEPTH_KM, KIM1983, "P"),
                     "TauP - yours, S (s)": base[1] - direct_time(x_st, DEPTH_KM, KIM1983, "S"),
                     f"P at {CHANGED_DEPTH_KM:g} km - at {DEPTH_KM:g} km (s)": changed[0] - base[0],
                     f"S at {CHANGED_DEPTH_KM:g} km - at {DEPTH_KM:g} km (s)": changed[1] - base[1]})
print(runs.round(4).to_string(index=False))
runs.to_csv(Path(RESULT_DIR) / "M30_taup_runs.csv", index=False)

x_line = np.linspace(1.0, 150.0, 60)
fig, ax = plt.subplots(figsize=(6.0, 3.8))
for depth, style in ((5.0, ":"), (DEPTH_KM, "-"), (25.0, "--")):
    tp_line, _ = taup_times("kim1983", depth, x_line)
    ax.plot(x_line, tp_line - direct_time(x_line, depth, KIM1983, "P"), color="k", ls=style, lw=1.2,
            label=f"Source at {depth:g} km")
ax.axhline(0.0, color="0.6", lw=0.8)
ax.set_xlabel("Epicentral distance (km)")
ax.set_ylabel("TauP P - your P (s)")
ax.set_title("kim1983: TauP against your flat-layer times", fontsize=10)
fig.legend(loc="upper center", ncol=3, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(Path(FIGURE_DIR) / "taup_minus_flat.png", dpi=150)

**Explain:** TauP and your function use the same layers. Why are TauP's times slightly earlier, and does the
difference matter next to your picking errors?

Look at the last two columns of the table: the shallower source moves the times most at the nearest station.
Only stations close to the epicenter tell you the depth well (M31).

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M30")

**Optional:**

1. In explorer E4, move the kim1983 source from 14 to 16 km, just below the 15-km boundary. What happens to the
   first arrival at 70-100 km?
2. Repeat task 4 with the source at 8 km and at 20 km. Which depth fits your picks best?

**Read more:** Shearer (2019), travel times in layers; Crotwell et al. (1999), TauP; Chapman (1973), Earth
flattening. The [reading list](../../../docs/reading_list.md) has the rest.

Next: M22 (detection and template matching), then M31, which locates the Sangju earthquake with your picks and
your travel-time functions.